# 하이퍼파라미터 최적화 분석

## 목적
- Optuna TPE 탐색 과정 상세 분석
- 하이퍼파라미터 중요도 분석
- 파라미터 조합별 성능 분석
- 최적 설정 도출

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances
from optuna.visualization import plot_parallel_coordinate, plot_contour
from pathlib import Path

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
sns.set_palette('husl')

BASE_DIR = Path('c:/Users/Public/Documents/DIK/deTACTer')
MODELS_DIR = BASE_DIR / 'results' / 'models'
VERSIONS = ['v5.5', 'v5.6', 'v5.7']

## 1. Study 로드

In [ ]:
studies = {'gru': {}, 'transformer': {}}

for model_type in ['gru', 'transformer']:
    for version in VERSIONS:
        db_path = MODELS_DIR / version / f'{model_type}_{version}.db'
        if db_path.exists():
            storage = f'sqlite:///{db_path}'
            study = optuna.load_study(study_name=f'{model_type}_{version}', storage=storage)
            studies[model_type][version] = study
            print(f"{model_type.upper()} {version}: {len(study.trials)} trials, Best: {study.best_value:.6f}")

## 2. 하이퍼파라미터 중요도 분석

In [ ]:
# GRU 하이퍼파라미터 중요도
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, version in enumerate(VERSIONS):
    if version in studies['gru']:
        study = studies['gru'][version]
        
        # Optuna visualization을 matplotlib figure로 변환
        try:
            importance = optuna.importance.get_param_importances(study)
            
            params = list(importance.keys())
            values = list(importance.values())
            
            axes[idx].barh(params, values)
            axes[idx].set_xlabel('Importance')
            axes[idx].set_title(f'GRU {version} 파라미터 중요도')
            axes[idx].grid(True, alpha=0.3, axis='x')
        except:
            axes[idx].text(0.5, 0.5, 'Insufficient data', ha='center', va='center')

plt.tight_layout()
plt.savefig(MODELS_DIR / 'gru_param_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Transformer 하이퍼파라미터 중요도
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, version in enumerate(VERSIONS):
    if version in studies['transformer']:
        study = studies['transformer'][version]
        
        try:
            importance = optuna.importance.get_param_importances(study)
            
            params = list(importance.keys())
            values = list(importance.values())
            
            axes[idx].barh(params, values)
            axes[idx].set_xlabel('Importance')
            axes[idx].set_title(f'Transformer {version} 파라미터 중요도')
            axes[idx].grid(True, alpha=0.3, axis='x')
        except:
            axes[idx].text(0.5, 0.5, 'Insufficient data', ha='center', va='center')

plt.tight_layout()
plt.savefig(MODELS_DIR / 'transformer_param_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. 파라미터 분포 분석

In [ ]:
# GRU 파라미터 분포 (v5.5 예시)
if 'v5.5' in studies['gru']:
    study = studies['gru']['v5.5']
    
    # Trial 데이터 추출
    trials_df = study.trials_dataframe()
    completed_trials = trials_df[trials_df['state'] == 'COMPLETE']
    
    # 주요 파라미터 분포
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    params_to_plot = ['params_hidden_dim', 'params_num_layers', 'params_dropout', 
                      'params_lr', 'params_batch_size', 'value']
    
    for idx, param in enumerate(params_to_plot):
        if param in completed_trials.columns:
            if param == 'value':
                axes[idx].hist(completed_trials[param], bins=30, alpha=0.7, edgecolor='black')
                axes[idx].set_xlabel('Validation Loss')
            else:
                axes[idx].hist(completed_trials[param], bins=20, alpha=0.7, edgecolor='black')
                axes[idx].set_xlabel(param.replace('params_', ''))
            
            axes[idx].set_ylabel('Frequency')
            axes[idx].grid(True, alpha=0.3)
    
    plt.suptitle('GRU v5.5 파라미터 분포', fontsize=14, y=1.00)
    plt.tight_layout()
    plt.savefig(MODELS_DIR / 'gru_v5.5_param_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. 파라미터 간 상관관계

In [ ]:
# GRU 파라미터 상관관계 (v5.5)
if 'v5.5' in studies['gru']:
    study = studies['gru']['v5.5']
    trials_df = study.trials_dataframe()
    completed_trials = trials_df[trials_df['state'] == 'COMPLETE']
    
    # 수치형 파라미터만 선택
    param_cols = [col for col in completed_trials.columns if col.startswith('params_')]
    param_cols.append('value')
    
    corr_data = completed_trials[param_cols].copy()
    
    # 상관계수 계산
    correlation = corr_data.corr()
    
    # 히트맵
    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                xticklabels=[c.replace('params_', '') for c in correlation.columns],
                yticklabels=[c.replace('params_', '') for c in correlation.columns])
    plt.title('GRU v5.5 파라미터 상관관계')
    plt.tight_layout()
    plt.savefig(MODELS_DIR / 'gru_v5.5_correlation.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. Learning Rate vs Performance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# GRU
for version in VERSIONS:
    if version in studies['gru']:
        study = studies['gru'][version]
        trials_df = study.trials_dataframe()
        completed = trials_df[trials_df['state'] == 'COMPLETE']
        
        axes[0].scatter(completed['params_lr'], completed['value'], 
                       label=version, alpha=0.6, s=30)

axes[0].set_xlabel('Learning Rate')
axes[0].set_ylabel('Validation Loss')
axes[0].set_xscale('log')
axes[0].set_title('GRU: Learning Rate vs Performance')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Transformer
for version in VERSIONS:
    if version in studies['transformer']:
        study = studies['transformer'][version]
        trials_df = study.trials_dataframe()
        completed = trials_df[trials_df['state'] == 'COMPLETE']
        
        axes[1].scatter(completed['params_lr'], completed['value'], 
                       label=version, alpha=0.6, s=30)

axes[1].set_xlabel('Learning Rate')
axes[1].set_ylabel('Validation Loss')
axes[1].set_xscale('log')
axes[1].set_title('Transformer: Learning Rate vs Performance')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(MODELS_DIR / 'lr_vs_performance.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. 최적 파라미터 요약

In [ ]:
# 모든 버전의 최적 파라미터 정리
summary = []

for model_type in ['gru', 'transformer']:
    for version in VERSIONS:
        if version in studies[model_type]:
            study = studies[model_type][version]
            row = {
                'Model': model_type.upper(),
                'Version': version,
                'Best Loss': f"{study.best_value:.6f}",
                **{k: v for k, v in study.best_params.items()}
            }
            summary.append(row)

summary_df = pd.DataFrame(summary)
print("\n" + "="*100)
print("최적 하이퍼파라미터 요약")
print("="*100)
print(summary_df.to_string(index=False))
print("="*100)

# CSV로 저장
summary_df.to_csv(MODELS_DIR / 'best_hyperparameters.csv', index=False, encoding='utf-8-sig')
print(f"\n저장 완료: {MODELS_DIR / 'best_hyperparameters.csv'}")

## 7. 결론

### 주요 인사이트
1. **가장 중요한 하이퍼파라미터**: (위 중요도 분석 참조)
2. **최적 Learning Rate 범위**: (위 scatter plot 참조)
3. **모델 크기 vs 성능**: (hidden_dim, num_layers 분석)
4. **버전별 차이**: 데이터 크기에 따른 최적 설정 변화